In [4]:
import math

# --- PARAMETER SKEMA ---
W = 3  # Ukuran blok (w)
MODULUS = 511  # Modulo untuk fungsi hash (511)
PESAN_BITS = "101110100"  # Pesan m (9 bit)
# 5 nilai kunci privat pertama yang diperlukan (sesuai L=5)
KUNCI_PRIVAT_X = [184, 245, 20, 60, 311] 

# --- FUNGSI HASH ---
def h(x):
    """Fungsi hash h(x) = x^2 mod 511"""
    return (x * x) % MODULUS

def h_iterasi(x, k):
    """Menghitung h^k(x)"""
    result = x
    for _ in range(k):
        result = h(result)
    return result

# --- PERHITUNGAN PARAMETER L ---
n = len(PESAN_BITS)
L1 = math.ceil(n / W)  # 3 blok
MAX_BLOCK_VAL = 2**W - 1  # 7
MAX_CHECKSUM = L1 * MAX_BLOCK_VAL  # 21
N_C = MAX_CHECKSUM.bit_length()  # 5 bit
L2 = math.ceil(N_C / W)  # 2 blok
L_TOTAL = L1 + L2  # 5 blok

print("--- PARAMETER SKEMA ---")
print(f"Total Blok Tanda Tangan (L): {L_TOTAL} (3 Pesan + 2 Checksum)")
print(f"Iterasi Maksimum (2^w - 1): {MAX_BLOCK_VAL}\n")

# --- FUNGSI UTAMA ---

def hitung_blok_Z(m_bits):
    """Menghitung blok pesan dan blok checksum (Z)"""
    
    # 1. BLOK PESAN (L1)
    m_blocks = []
    checksum = 0
    for i in range(L1):
        start = i * W
        end = start + W
        block_str = m_bits[start:end]
        block_val = int(block_str, 2)
        m_blocks.append(block_val)
        checksum += block_val
    
    # 2. BLOK CHECKSUM (L2)
    # Format checksum ke biner, pastikan panjangnya N_C=5 bit
    checksum_binary = format(checksum, f'0{N_C}b') 
    
    c_blocks = [0,6]
    # for i in range(L2):
    #     start = i * W
    #     end = start + W
    #     block_str = checksum_binary[start:end]
    #     block_val = int(block_str, 2)
    #     c_blocks.append(block_val)
        
    return m_blocks + c_blocks, checksum

def buat_kunci_publik(X):
    """Y = h^(2^w - 1)(x_i)"""
    Y = [h_iterasi(x, MAX_BLOCK_VAL) for x in X[:L_TOTAL]]
    return Y

def buat_tanda_tangan(X, Z):
    """sigma = h^(z_i)(x_i)"""
    sigma = [h_iterasi(X[i], Z[i]) for i in range(L_TOTAL)]
    return sigma

def verifikasi(sigma, Y, Z):
    """Verifikasi: Cek apakah h^(2^w - 1 - z_i)(sigma_i) == y_i"""
    y_verifikasi = []
    is_valid = True
    
    for i in range(L_TOTAL):
        # Iterasi yang dibutuhkan: k = 2^w - 1 - z_i
        k = MAX_BLOCK_VAL - Z[i]
        y_prime = h_iterasi(sigma[i], k)
        y_verifikasi.append(y_prime)
        
        if y_prime != Y[i]:
            is_valid = False
            
    return is_valid, y_verifikasi

# --- EKSEKUSI ---

# Langkah 1: Hitung Blok Z
Z, C = hitung_blok_Z(PESAN_BITS)
print(f"Blok Pesan (m'): {Z[:L1]}")
print(f"Blok Checksum (C'): {Z[L1:]}")
print(f"Nilai Hash/Blok Z: {Z}\n")

# Langkah 2: Buat Kunci Publik (Jawaban II)
Y = buat_kunci_publik(KUNCI_PRIVAT_X)
print(f"=== KUNCI PUBLIK (Y) ===")
print(f"Kunci Publik Y: {Y}") # Hasil: [499, 500, 479, 499, 435]
print("-" * 25)

# Langkah 3: Buat Tanda Tangan (Jawaban III)
sigma = buat_tanda_tangan(KUNCI_PRIVAT_X, Z)
print(f"=== TANDA TANGAN (sigma) ===")
print(f"Tanda Tangan sigma: {sigma}") # Hasil: [452, 388, 48, 190, 302]
print("-" * 25)

# Langkah 4 & 5: Verifikasi
is_valid, y_verifikasi = verifikasi(sigma, Y, Z)
print(f"=== PROSES VERIFIKASI ===")
print(f"Kunci Verifikasi (Y'): {y_verifikasi}")
print(f"Kunci Publik Asli (Y): {Y}")
print("-" * 25)
if is_valid:
    print("STATUS: TANDA TANGAN VALID.")
else:
    print("❌ STATUS: TANDA TANGAN TIDAK VALID.")

--- PARAMETER SKEMA ---
Total Blok Tanda Tangan (L): 5 (3 Pesan + 2 Checksum)
Iterasi Maksimum (2^w - 1): 7

Blok Pesan (m'): [5, 6, 4]
Blok Checksum (C'): [0, 6]
Nilai Hash/Blok Z: [5, 6, 4, 0, 6]

=== KUNCI PUBLIK (Y) ===
Kunci Publik Y: [235, 294, 442, 37, 296]
-------------------------
=== TANDA TANGAN (sigma) ===
Tanda Tangan sigma: [221, 105, 274, 60, 221]
-------------------------
=== PROSES VERIFIKASI ===
Kunci Verifikasi (Y'): [235, 294, 442, 37, 296]
Kunci Publik Asli (Y): [235, 294, 442, 37, 296]
-------------------------
STATUS: TANDA TANGAN VALID.


In [11]:
# --- KONFIGURASI DAN PARAMETER ---
MODULUS = 512
DAUN_V0 = [411, 245, 44, 192, 376, 199, 418, 53] # v0[0] sampai v0[7]

# --- FUNGSI MATEMATIKA ---

def f(x):
    """
    Fungsi hash sesuai soal: f(x) = x^2 + 7x (mod 512)
    """
    # Catatan: Kita hitung x^2 + 7x dulu, baru dimodulo 512
    val = (x**2) + (7*x)
    return val % MODULUS

def gabung_node(kiri, kanan):
    """
    Aturan penggabungan: h(xi) = f(kiri + kanan)
    """
    jumlah = kiri + kanan
    # Terapkan f pada jumlah tersebut
    parent = f(jumlah)
    return parent

# --- MEMBANGUN TREE ---

def bangun_merkle_tree(leaves):
    tree = []
    tree.append(leaves) # Level 0
    
    current_level = leaves
    
    # Loop sampai kita mencapai Root (hanya 1 elemen tersisa)
    while len(current_level) > 1:
        next_level = []
        # Iterasi setiap pasangan (step 2)
        for i in range(0, len(current_level), 2):
            kiri = current_level[i]
            kanan = current_level[i+1]
            
            parent = gabung_node(kiri, kanan)
            next_level.append(parent)
            
        tree.append(next_level)
        current_level = next_level
        
    return tree

# --- FUNGSI MENCARI JALUR AUTENTIKASI ---

def get_auth_path(tree, index_daun):
    path = []
    num_levels = len(tree) - 1 # Minus root level
    
    idx = index_daun
    
    for level in range(num_levels):
        # Tentukan apakah node kita di kiri atau kanan
        # Jika genap -> node di kiri, sibling di kanan (idx + 1)
        # Jika ganjil -> node di kanan, sibling di kiri (idx - 1)
        if idx % 2 == 0:
            sibling_idx = idx + 1
        else:
            sibling_idx = idx - 1
            
        sibling_val = tree[level][sibling_idx]
        path.append(sibling_val)
        
        # Naik ke level berikutnya (index parent adalah floor division 2)
        idx = idx // 2
        
    return path

# --- EKSEKUSI UTAMA ---

# 1. Bangun Pohon
merkle_tree = bangun_merkle_tree(DAUN_V0)

# 2. Tampilkan Struktur Pohon untuk verifikasi manual
print("--- STRUKTUR MERKLE TREE ---")
for i, level in enumerate(merkle_tree):
    print(f"Level {i}: {level}")

# 3. Jawaban Soal I: Kunci Publik (Root)
root = merkle_tree[-1][0]
print("\n--- JAWABAN I: KUNCI PUBLIK ---")
print(f"Root (Kunci Publik) Alice adalah: {root}")

# 4. Jawaban Soal II: Tanda Tangan (Jalur Autentikasi)
# Asumsi Alice menandatangani pesan pertama (Index 0)
index_pesan = 0 
auth_path = get_auth_path(merkle_tree, index_pesan)

print("\n--- JAWABAN II: TANDA TANGAN (BAGIAN PATH) ---")
print(f"Misalkan Alice menandatangani pesan pada Index {index_pesan}.")
print(f"Tanda tangan digital:")
print(f"{auth_path}")
print("(Urutan: Sibling Level 0 -> Sibling Level 1 -> Sibling Level 2)")

--- STRUKTUR MERKLE TREE ---
Level 0: [411, 245, 44, 192, 376, 199, 418, 53]
Level 1: [240, 4, 314, 370]
Level 2: [316, 68]
Level 3: [128]

--- JAWABAN I: KUNCI PUBLIK ---
Root (Kunci Publik) Alice adalah: 128

--- JAWABAN II: TANDA TANGAN (BAGIAN PATH) ---
Misalkan Alice menandatangani pesan pada Index 0.
Tanda tangan digital:
[245, 4, 68]
(Urutan: Sibling Level 0 -> Sibling Level 1 -> Sibling Level 2)


In [10]:
print((3+ 2 * 5) % 12)

1
